[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/photoelasticity/photoelasticity.ipynb)

# Stress seen through polarizers

Pull a transparent plate containing a circular hole: stress crowds around the rim and makes the material birefringent. A stress extensor maps a surface normal to traction. Its principal directions and their difference set a rotor that turns the light's polarization, a point on the Poincaré sphere; polarizers turn that change into visible fringes.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, concatenate
from numga.algebras import VGA3D
from examples.optics.photoelasticity import render
from examples.optics.photoelasticity.core import square_grid

In [ ]:
ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# The plate lies in x y; the stress maps a surface normal in it to the traction across it.
Planar = ga.gatype.from_blades("x y")
Stress = ga.gatype((Planar, Planar))                         # Planar <- Planar
# Light's polarization is a unit vector on the Poincaré sphere, and the plate turns that sphere.
Polarization = ga.gatype.vector()
Retarder = ga.gatype.rotor()

In [ ]:
radius = 10.0                                       # mm, hole radius
extent = 30.0                                       # mm, sampling half-width
radial_samples = 200
angular_samples = 512
tension = 30.0                                              # MPa, pulling along x

positions = square_grid(radius, extent, radial_samples, angular_samples)          # [radii, angles + 1] Planar
# Average identity and reflection in x: keep axial traction, cancel transverse traction.
remote = tension * (Planar + (mv.x >> Planar)) / 2                                # [] Planar <- Planar
distance_squared = positions.squared()                                            # [radii, angles + 1] Scalar
radial = positions / distance_squared.square_root()                               # [radii, angles + 1] Planar
tangent = mv.xy | radial                                                          # [radii, angles + 1] Planar
radius_ratio = radius**2 / distance_squared                                       # [radii, angles + 1] Scalar

# Resolve the remote stress along the local radial and tangential directions.
mean = remote.trace() / 2                                                         # [] Scalar
anisotropy = (radial | remote(radial)) - mean                                     # [radii, angles + 1] Scalar
shear = radial | remote(tangent)                                                  # [radii, angles + 1] Scalar
# Kirsch's exterior solution redistributes traction around a circular, unloaded rim.
# At the rim radius_ratio is one: both radial and sliding traction vanish.
radial_stress = (mean * (1 - radius_ratio)
                 + anisotropy * (1 - 4 * radius_ratio + 3 * radius_ratio.squared()))  # [radii, angles + 1] Scalar
tangential_stress = (mean * (1 + radius_ratio)
                     - anisotropy * (1 + 3 * radius_ratio.squared()))             # [radii, angles + 1] Scalar
shear_stress = shear * (1 + 2 * radius_ratio - 3 * radius_ratio.squared())        # [radii, angles + 1] Scalar
# A direction paired with the open Planar reads a normal component; multiplying by
# a direction turns that reading into traction. Summing the dyads builds the stress map.
stress = (radial_stress * radial * (radial | Planar)
          + tangential_stress * tangent * (tangent | Planar)
          + shear_stress * (radial * (tangent | Planar) + tangent * (radial | Planar)))  # [radii, angles + 1] Planar <- Planar

principal, directions = stress.eigh()                                             # [radii, angles + 1, modes] Scalar, Planar
# The two stresses differ most strongly at the top and bottom of the hole.
difference = principal[..., 1] - principal[..., 0]                                # [radii, angles + 1] Scalar
render.draw_stress(positions, difference, radius, tension);

Light polarized along one principal direction is delayed more than light along the other. Any other polarization is a mix of the two, and the growing delay between them carries it through elliptical states and back. On the Poincaré sphere, linear polarizations lie on the equator at twice their angle in the plate, so that a direction and its opposite are the same state, and the two circular polarizations sit at the poles. The plate turns the whole sphere about the state polarized along a principal direction, by the retardance: the phase that the principal stress difference builds up through the thickness. A polarizer passes the share of light given by Malus's law, half of one plus the inner product of its own state with the light's. Crossed linear polarizers go dark wherever a principal direction lines up with them, since their state then lies on the axis of the turn. Circular light starts at a pole, the same distance from every point on the equator, so a circular polariscope shows only the fringes of principal stress difference.

In the Jones calculus this reads as the unitary $\operatorname{diag}(e^{i\delta/2}, e^{-i\delta/2})$ in the principal frame, acting on a complex two-component field, and an analyser reading $|\langle a|\psi\rangle|^2$.

In [ ]:
thickness = 1.0                                            # mm
wavelength = 550e-6                                        # mm, monochromatic green light
stress_optic = 5.5e-5                                      # inverse MPa, illustrative sensitivity
polarizer_angles = np.deg2rad([0, 22.5])
stress_phase = 2 * np.pi * stress_optic * thickness / wavelength


def polarization(direction: Planar) -> Polarization:
    # Reflecting x in a direction turns it by twice the direction's angle: the sphere's equator.
    return direction >> mv.x


# The mean stress delays both principal directions alike and turns nothing; their difference sets the retardance.
retardance = stress_phase * difference                                            # [radii, angles + 1] Scalar
# The axis of the turn is the state polarized along the second principal direction.
axis = polarization(directions[..., 1])                                           # [radii, angles + 1] Polarization
retarder = (axis.dual() * (retardance / 2)).exp()                                 # [radii, angles + 1] Retarder

# Two crossed linear polariscopes, turned together, and a circular one at the sphere's pole.
linear = polarization((mv.xy * (-polarizer_angles / 2)).exp() >> mv.x)            # [linear views] Polarization
incident = concatenate((linear, mv.z[None]))                                      # [views] Polarization
# Each analyser passes the opposite state: without stress, every view is dark.
analyser = -incident                                                              # [views] Polarization
exits = retarder >> incident[:, None, None]                                       # [views, radii, angles + 1] Polarization
intensities = (1 + (analyser[:, None, None] | exits)) / 2                         # [views, radii, angles + 1] Scalar
render.draw_polariscope(positions, intensities, radius);

In [ ]:
frames = 80
duration_ms = 90
# Slowly apply the full tension and release it; this is a sequence of static equilibria.
loads = (1 - np.cos(np.linspace(0, 2 * np.pi, frames, endpoint=False))) / 2

# Scaling the load keeps the principal directions, so the axis of each turn stays put
# and only its angle grows: the bivector is built once, and each load scales it.
half_turn = axis.dual() * (retardance / 2)                                        # [radii, angles + 1] Bivector


def loading() -> Iterator[Scalar]:
    for load in loads:
        exits = (half_turn * load).exp() >> mv.z                                  # [radii, angles + 1] Polarization
        yield (1 + (-mv.z | exits)) / 2                                           # [radii, angles + 1] Scalar


render.inline(render.animate_polariscope(positions, loading(), radius), duration_ms)

The square is a window into an infinite plate, not a free outer edge. The model uses small-strain isotropic elasticity, uniform stress through the thickness, and a linear stress–optic law; light enters normally, with reflection, absorption, ray bending, and diffraction neglected. The optical coefficient is illustrative rather than calibrated to a particular material. [MIT's hole-in-a-plate experiment](https://ocw.mit.edu/courses/2-002-mechanics-and-materials-ii-spring-2004/09e8137971e754b1b7290a33a100bb05_lab_4_s04.pdf) gives the stress solution and experimental context; its [experimental mechanics notes](https://web.mit.edu/course/3/3.11/www/modules/expt.pdf) introduce photoelastic measurement.